In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

#
# Change "Questionnnair" -> "Questionnaire"
# if your folder uses the correct spelling.
#
DATA_DIR = (
    BASE_DIR
    / "data"
    / "2015_2016"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "HUQ_I.xpt"
csv_file = DATA_DIR / "HUQ_I_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve completely untouched pandas-decoded XPT.
#
source_df = df.copy()


print("=" * 100)
print("NHANES 2015-2016 HUQ_I XPT -> CSV")
print("=" * 100)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "HUQ010",
    "HUQ020",
    "HUQ030",
    "HUQ041",
    "HUQ051",
    "HUQ061",
    "HUQ071",
    "HUD080",
    "HUQ090"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "HUQ_I column names/order differ.\n\n"
        f"Expected:\n{EXPECTED_COLUMNS}\n\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != 9971:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 10:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 9,971 rows x 10 columns."
)


# ============================================================
# 4. FIELD TYPE PLAN
# ============================================================

#
# Direct inspection of attached HUQ_I:
#
#     genuine decimal variables = NONE
#     string variables          = NONE
#
#
# All 10 released fields contain only whole-number:
#
#     SEQN      -> respondent identifier
#
#     HUQ...    -> response categories / counts
#
#     HUD080    -> whole-number admission category
#
#
# Therefore all 10 variables can safely become
# nullable Int64 AFTER the validated HUQ051 zero repair.
#
#
# IMPORTANT:
#
# This applies ONLY to HUQ_I.
#
# It does NOT change your overall NHANES fidelity rule:
#
#     Dietary genuine decimals
#     Laboratory continuous results
#     Examination measurements
#     survey/sample weights
#
# must remain exactly AS IS.
#
# No blanket rounding should ever be applied.
#

DECIMAL_COLUMNS = []

STRING_COLUMNS = []

INTEGER_COLUMNS = EXPECTED_COLUMNS.copy()


if len(INTEGER_COLUMNS) != 10:

    raise ValueError(
        "Expected exactly 10 whole-number/code fields."
    )


print(
    "\n--- FIELD TYPE PLAN ---"
)

print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)

print(
    "String fields:",
    STRING_COLUMNS
)

print(
    "Whole-number/code fields:",
    INTEGER_COLUMNS
)


# ============================================================
# 5. CHECK KNOWN SAS XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas/SAS XPORT representation of numeric zero:
#
#     5.397605346934028e-79
#
#
# IMPORTANT:
#
# Never globally replace this value before validating
# the exact variable/count pattern.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_cols != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected all 10 HUQ_I variables "
        "to load numerically."
    )


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


tiny_variable_count = int(
    tiny_counts.gt(0)
    .sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print(
    "\n--- ORIGINAL XPT ZERO CHECK ---"
)

print(
    "Ordinary numeric zeros before correction:",
    f"{true_zero_total:,}"
)

print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)

print(
    "Variables containing tiny artifacts:",
    tiny_variable_count
)


print(
    "\nTiny values by variable:"
)

print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 6. VALIDATE EXACT TINY-ZERO PATTERN
# ============================================================

#
# Direct inspection of attached HUQ_I.xpt:
#
#     HUQ051 = 1,421 tiny values
#
# TOTAL = 1,421
#
#
# CDC codebook independently reports:
#
# HUQ051:
#
#     0 = None
#
#     Count = 1,421
#
#
# This provides a very strong validation that the
# 1,421 tiny values represent legitimate numeric zero.
#

EXPECTED_TINY_COUNTS = {
    "HUQ051": 1421
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: Tiny-value pattern differs from "
        "validated HUQ_I attachment.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{actual_tiny_counts}"
    )


if tiny_total != 1421:

    raise ValueError(
        f"Unexpected total tiny-value count: "
        f"{tiny_total:,}"
    )


if tiny_variable_count != 1:

    raise ValueError(
        "Expected tiny artifacts in exactly 1 variable."
    )


if true_zero_total != 0:

    raise ValueError(
        "Unexpected ordinary zeros existed "
        "before correction."
    )


print(
    "\nPASS: Exact HUQ051 tiny-zero pattern "
    "validated."
)


# ============================================================
# 7. RESTORE ONLY VALIDATED HUQ051 ZEROS
# ============================================================

#
# NEVER:
#
#     df.replace(TINY_VALUE, 0)
#
# globally.
#
# Repair only HUQ051 after validating:
#
#     variable
#     exact count
#     CDC legitimate zero category
#

huq051_zero_mask = (
    df["HUQ051"]
    .eq(TINY_VALUE)
)


huq051_zero_count = int(
    huq051_zero_mask.sum()
)


if huq051_zero_count != 1421:

    raise ValueError(
        "Expected exactly 1,421 tiny HUQ051 values."
    )


df.loc[
    huq051_zero_mask,
    "HUQ051"
] = 0


print(
    "\nPASS: 1,421 validated HUQ051 "
    "values restored to numeric zero."
)


# ============================================================
# 8. CREATE EXPECTED CORRECTED SOURCE COPY
# ============================================================

#
# expected_df becomes our exact fidelity reference.
#
# The ONLY numerical modification is:
#
#     HUQ051 tiny value -> 0
#

expected_df = source_df.copy()


expected_df.loc[
    expected_df["HUQ051"]
    .eq(TINY_VALUE),
    "HUQ051"
] = 0


# ============================================================
# 9. CONFIRM NO TINY VALUES REMAIN
# ============================================================

remaining_tiny = int(
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nTiny values remaining:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain "
        "after HUQ051 correction."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 10. DETECT GENUINE DECIMAL VARIABLES
# ============================================================

#
# Run BEFORE converting fields to Int64.
#
#
# np.round() here is ONLY used to determine whether
# an observation has a fractional component.
#
# It NEVER changes source data.
#

print(
    "\n--- GENUINE DECIMAL DETECTION ---"
)


fractional_columns = []

fractional_counts = {}


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        fractional_counts[col] = 0
        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    fractional_count = int(
        fractional_mask.sum()
    )


    fractional_counts[col] = fractional_count


    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )


    if fractional_count > 0:

        fractional_columns.append(
            col
        )


if fractional_columns:

    raise ValueError(
        "STOP: Unexpected genuine decimal variables "
        "found in HUQ_I.\n\n"
        "These values will NOT be rounded:\n"
        + str(
            fractional_columns
        )
    )


print(
    "\nPASS: No genuine decimal variables "
    "found in HUQ_I."
)


# ============================================================
# 11. CLEAN AND VALIDATE SEQN
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    seqn_values,
    np.round(seqn_values),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print(
    "\n--- SEQN ---"
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 83732:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 93702:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "Missing SEQN found."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN found."
    )


print(
    "PASS: SEQN 83732-93702 validated."
)


# ============================================================
# 12. VALIDATE ALL 10 FIELDS ARE WHOLE-NUMBER
# ============================================================

#
# Every populated value must be mathematically whole-number
# before Int64 conversion.
#

for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    whole_mask = np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if not whole_mask.all():

        raise ValueError(
            f"{col} unexpectedly contains "
            "fractional values."
        )


print(
    "PASS: All 10 HUQ_I variables contain only "
    "whole-number values/codes."
)


# ============================================================
# 13. CONVERT ALL FIELDS TO NULLABLE Int64
# ============================================================

#
# NO .round() is used during actual conversion.
#

for col in INTEGER_COLUMNS:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All 10 HUQ_I variables converted "
    "to nullable Int64 without rounding."
)


# ============================================================
# 14. HELPER - EXACT FREQUENCY CHECK
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )


    print(
        f"\n--- {column} ---"
    )

    print(
        "Observed:",
        actual_values
    )

    print(
        "Missing:",
        f"{actual_missing:,}"
    )


    if actual_values != expected_values:

        raise ValueError(
            f"{column} frequencies differ.\n\n"
            f"Expected:\n{expected_values}\n\n"
            f"Actual:\n{actual_values}"
        )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {actual_missing:,}"
        )


    print(
        f"PASS: {column} matches CDC."
    )


# ============================================================
# 15. CDC CHECK - HUQ010
# ============================================================

#
# General health condition:
#
#     1 = Excellent       2,561
#     2 = Very good       2,658
#     3 = Good            3,001
#     4 = Fair            1,424
#     5 = Poor              317
#     7 = Refused             3
#     9 = Don't know          7
#
# Missing = 0
#

check_frequency(
    "HUQ010",
    {
        1: 2561,
        2: 2658,
        3: 3001,
        4: 1424,
        5: 317,
        7: 3,
        9: 7
    },
    0
)


# ============================================================
# 16. CDC CHECK - HUQ020
# ============================================================

#
# Compared with 1 year ago:
#
#     1 = Better            1,965
#     2 = Worse               653
#     3 = About the same    6,954
#     9 = Don't know            3
#
# Missing = 396
#
#
# CDC states HUQ020 was asked only for age >= 1 year.
#

check_frequency(
    "HUQ020",
    {
        1: 1965,
        2: 653,
        3: 6954,
        9: 3
    },
    396
)


# ============================================================
# 17. CDC CHECK - HUQ030
# ============================================================

#
# Routine place to go for healthcare:
#
#     1 = Yes                         8,563
#     2 = There is no place           1,340
#     3 = More than one place            68
#
# Missing = 0
#

check_frequency(
    "HUQ030",
    {
        1: 8563,
        2: 1340,
        3: 68
    },
    0
)


# ============================================================
# 18. CDC CHECK - HUQ041
# ============================================================

#
# Type place most often go for healthcare:
#
#     1 = Clinic / health center                2,979
#     2 = Doctor's office / HMO                 5,093
#     3 = Hospital emergency room                 277
#     4 = Hospital outpatient department          148
#     5 = Some other place                        104
#     6 = Doesn't go to one place most often       30
#
# Missing = 1,340
#

check_frequency(
    "HUQ041",
    {
        1: 2979,
        2: 5093,
        3: 277,
        4: 148,
        5: 104,
        6: 30
    },
    1340
)


# ============================================================
# 19. HUQ030 -> HUQ041 ROUTING CHECK
# ============================================================

#
# HUQ041 should be asked when:
#
#     HUQ030 = 1
#     OR
#     HUQ030 = 3
#
#
# HUQ030 = 2 skips HUQ041.
#

huq041_expected = (
    df["HUQ030"]
    .isin(
        [1, 3]
    )
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


huq041_present = (
    df["HUQ041"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


huq041_routing_mismatch = int(
    np.sum(
        huq041_expected
        !=
        huq041_present
    )
)


print(
    "\n--- HUQ030 / HUQ041 ROUTING ---"
)

print(
    "Expected HUQ041:",
    int(
        huq041_expected.sum()
    )
)

print(
    "HUQ041 populated:",
    int(
        huq041_present.sum()
    )
)

print(
    "Routing mismatches:",
    huq041_routing_mismatch
)


if huq041_routing_mismatch != 0:

    raise ValueError(
        "HUQ041 availability differs from "
        "HUQ030 routing."
    )


print(
    "PASS: HUQ030/HUQ041 routing validated."
)


# ============================================================
# 20. CDC CHECK - HUQ051
# ============================================================

#
# Number of times received healthcare over past year:
#
#     0  = None             1,421
#     1  = 1                2,049
#     2  = 2 to 3           3,140
#     3  = 4 to 5           1,487
#     4  = 6 to 7             699
#     5  = 8 to 9             271
#     6  = 10 to 12           437
#     7  = 13 to 15           133
#     8  = 16 or more         304
#     77 = Refused              2
#     99 = Don't know          28
#
# Missing = 0
#
#
# The 1,421 zero values are exactly the values repaired
# from the XPORT tiny artifact.
#

check_frequency(
    "HUQ051",
    {
        0: 1421,
        1: 2049,
        2: 3140,
        3: 1487,
        4: 699,
        5: 271,
        6: 437,
        7: 133,
        8: 304,
        77: 2,
        99: 28
    },
    0
)


# ============================================================
# 21. HUQ051 ZERO VALIDATION
# ============================================================

huq051_zero_final = int(
    df["HUQ051"]
    .eq(0)
    .sum()
)


print(
    "\n--- HUQ051 ZERO VALIDATION ---"
)

print(
    "HUQ051 zero values:",
    f"{huq051_zero_final:,}"
)


if huq051_zero_final != 1421:

    raise ValueError(
        "HUQ051 final zero count differs."
    )


print(
    "PASS: Exactly 1,421 legitimate "
    "HUQ051=0 responses restored."
)


# ============================================================
# 22. CDC CHECK - HUQ061
# ============================================================

#
# How long since last healthcare visit:
#
#     1 = <= 6 months                    114
#     2 = >6 months <=1 year             104
#     3 = >1 year <=2 years              508
#     4 = >2 years <=5 years             409
#     5 = >5 years                       221
#     6 = Never                           59
#     99 = Don't know                      6
#
# Missing = 8,550
#

check_frequency(
    "HUQ061",
    {
        1: 114,
        2: 104,
        3: 508,
        4: 409,
        5: 221,
        6: 59,
        99: 6
    },
    8550
)


# ============================================================
# 23. HUQ051 -> HUQ061 ROUTING CHECK
# ============================================================

#
# HUQ061 is asked exactly when:
#
#     HUQ051 = 0
#
# because the participant reported no healthcare visits
# during the past 12 months.
#
#
# Direct attachment:
#
#     HUQ051=0             = 1,421
#     HUQ061 nonmissing    = 1,421
#     routing mismatches   = 0
#

huq061_expected = (
    df["HUQ051"]
    .eq(0)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


huq061_present = (
    df["HUQ061"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


huq061_routing_mismatch = int(
    np.sum(
        huq061_expected
        !=
        huq061_present
    )
)


print(
    "\n--- HUQ051 / HUQ061 ROUTING ---"
)

print(
    "HUQ051=0:",
    int(
        huq061_expected.sum()
    )
)

print(
    "HUQ061 populated:",
    int(
        huq061_present.sum()
    )
)

print(
    "Routing mismatches:",
    huq061_routing_mismatch
)


if huq061_routing_mismatch != 0:

    raise ValueError(
        "HUQ061 availability does not match "
        "HUQ051=0 routing."
    )


print(
    "PASS: HUQ051/HUQ061 routing validated exactly."
)


# ============================================================
# 24. CDC CHECK - HUQ071
# ============================================================

#
# Overnight hospital patient during past year:
#
#     1 = Yes          869
#     2 = No         9,098
#     9 = Don't know    4
#
# Missing = 0
#

check_frequency(
    "HUQ071",
    {
        1: 869,
        2: 9098,
        9: 4
    },
    0
)


# ============================================================
# 25. CDC CHECK - HUD080
# ============================================================

#
# Number of different overnight hospital stays:
#
#     1 = 1              630
#     2 = 2              141
#     3 = 3               55
#     4 = 4               18
#     5 = 5               14
#     6 = 6 or more       10
#     99999 = Don't know   1
#
# Missing = 9,102
#

check_frequency(
    "HUD080",
    {
        1: 630,
        2: 141,
        3: 55,
        4: 18,
        5: 14,
        6: 10,
        99999: 1
    },
    9102
)


# ============================================================
# 26. HUQ071 -> HUD080 ROUTING CHECK
# ============================================================

#
# HUD080 should exist exactly when:
#
#     HUQ071 = 1
#

hud080_expected = (
    df["HUQ071"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


hud080_present = (
    df["HUD080"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


hud080_routing_mismatch = int(
    np.sum(
        hud080_expected
        !=
        hud080_present
    )
)


print(
    "\n--- HUQ071 / HUD080 ROUTING ---"
)

print(
    "HUQ071=1:",
    int(
        hud080_expected.sum()
    )
)

print(
    "HUD080 populated:",
    int(
        hud080_present.sum()
    )
)

print(
    "Routing mismatches:",
    hud080_routing_mismatch
)


if hud080_routing_mismatch != 0:

    raise ValueError(
        "HUD080 availability does not match "
        "HUQ071=1 routing."
    )


print(
    "PASS: HUQ071/HUD080 routing validated exactly."
)


# ============================================================
# 27. CDC CHECK - HUQ090
# ============================================================

#
# Seen/talked to mental health professional
# during past 12 months:
#
#     1 = Yes          788
#     2 = No         8,000
#     9 = Don't know    6
#
# Missing = 1,177
#
#
# CDC:
#
# HUQ090 is asked for participants age >= 4 years.
#
# Age is not contained in HUQ_I itself, so we validate
# the exact frequency/missing count here rather than
# inventing an age-routing test.
#

check_frequency(
    "HUQ090",
    {
        1: 788,
        2: 8000,
        9: 6
    },
    1177
)


# ============================================================
# 28. VALID RESPONSE CODE CHECK
# ============================================================

VALID_CODES = {

    "HUQ010": {
        1, 2, 3, 4, 5, 7, 9
    },

    "HUQ020": {
        1, 2, 3, 9
    },

    "HUQ030": {
        1, 2, 3
    },

    "HUQ041": {
        1, 2, 3, 4, 5, 6
    },

    "HUQ051": {
        0, 1, 2, 3, 4, 5, 6, 7, 8, 77, 99
    },

    "HUQ061": {
        1, 2, 3, 4, 5, 6, 99
    },

    "HUQ071": {
        1, 2, 9
    },

    "HUD080": {
        1, 2, 3, 4, 5, 6, 99999
    },

    "HUQ090": {
        1, 2, 9
    }
}


print(
    "\n--- VALID RESPONSE CODE CHECK ---"
)


for col, allowed_codes in VALID_CODES.items():

    observed_codes = set(
        df[col]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )


    unexpected = (
        observed_codes
        -
        allowed_codes
    )


    print(
        f"{col}:",
        sorted(
            observed_codes
        )
    )


    if unexpected:

        raise ValueError(
            f"{col} contains unexpected codes: "
            f"{sorted(unexpected)}"
        )


print(
    "PASS: No unexpected HUQ response codes."
)


# ============================================================
# 29. FINAL VALIDATED ZERO PATTERN
# ============================================================

#
# Before repair:
#
#     ordinary numeric zero count = 0
#
#
# After repair:
#
#     HUQ051 = 1,421 zeros
#
# and NO other variable should contain zero.
#

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        final_zero_counts[col] = zero_count


EXPECTED_ZERO_COUNTS = {
    "HUQ051": 1421
}


print(
    "\n--- FINAL ZERO PATTERN ---"
)

print(
    final_zero_counts
)


if final_zero_counts != EXPECTED_ZERO_COUNTS:

    raise ValueError(
        "Final zero pattern differs.\n\n"
        f"Expected:\n{EXPECTED_ZERO_COUNTS}\n\n"
        f"Actual:\n{final_zero_counts}"
    )


print(
    "PASS: Final zero pattern contains exactly "
    "1,421 HUQ051 zeros."
)


# ============================================================
# 30. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# The ONLY allowed numerical modification:
#
#     HUQ051 tiny XPORT artifact -> 0
#
# Every other value must remain numerically identical.
#

print(
    "\n--- VALUE FIDELITY BEFORE EXPORT ---"
)


pre_export_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(
            pre_export_failures
        )
    )


print(
    "PASS: All 10 HUQ_I variables match "
    "the exact corrected XPT values."
)


# ============================================================
# 31. EXPORT CSV
# ============================================================

#
# CRITICAL:
#
# NO:
#
#     df.round(...)
#
# NO:
#
#     float_format=
#
#
# HUQ_I itself contains no genuine decimals,
# but using the same safe export method maintains
# consistency with Dietary/Laboratory/Exam files.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 32. RE-READ CSV WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 33. EXACT CORRECTED XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- EXACT CORRECTED XPT -> CSV FIDELITY ---"
)


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        pd.to_numeric(
            roundtrip_df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "Corrected XPT -> CSV numeric differences:\n"
        + str(
            csv_failures
        )
    )


print(
    "PASS: 0 numeric differences across "
    "all 10 HUQ_I variables."
)


# ============================================================
# 34. MISSING-VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print(
    "\n--- MISSING VALUE FIDELITY ---"
)

print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 35. READ FINAL CSV AS RAW TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 36. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values
        .eq(TINY_VALUE)
        .sum()
    )


print(
    "\n--- FINAL TINY VALUE CHECK ---"
)

print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain in CSV."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 37. RAW CSV .0 CHECK
# ============================================================

#
# HUQ_I contains NO genuine decimal variables.
#
# Therefore checking all 10 variables for unwanted
# ".0" is safe for THIS specific file.
#

print(
    "\n--- RAW CSV .0 CHECK ---"
)


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' formatting found:\n"
        + str(
            dot_zero_errors
        )
    )


print(
    "PASS: No unnecessary '.0' "
    "in any HUQ_I variable."
)


# ============================================================
# 38. FINAL HUQ051 ZERO CHECK
# ============================================================

final_huq051 = pd.to_numeric(
    raw_csv_df["HUQ051"],
    errors="coerce"
)


final_huq051_zero = int(
    final_huq051
    .eq(0)
    .sum()
)


print(
    "\n--- FINAL HUQ051 ZERO CHECK ---"
)

print(
    "HUQ051=0:",
    f"{final_huq051_zero:,}"
)


if final_huq051_zero != 1421:

    raise ValueError(
        "Final HUQ051 zero count changed."
    )


print(
    "PASS: Final CSV contains exactly "
    "1,421 HUQ051 zero responses."
)


# ============================================================
# 39. FINAL CSV ZERO PATTERN
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    zero_count = int(
        values
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        csv_zero_counts[col] = zero_count


print(
    "\n--- FINAL CSV ZERO PATTERN ---"
)

print(
    csv_zero_counts
)


if csv_zero_counts != EXPECTED_ZERO_COUNTS:

    raise ValueError(
        "Final CSV zero pattern differs.\n\n"
        f"Expected:\n{EXPECTED_ZERO_COUNTS}\n\n"
        f"Actual:\n{csv_zero_counts}"
    )


print(
    "PASS: Final CSV zero pattern preserved."
)


# ============================================================
# 40. FINAL ROUTING CHECKS
# ============================================================

#
# Re-check the three strongest routing relationships
# after CSV export.
#

final_huq030 = pd.to_numeric(
    raw_csv_df["HUQ030"]
    .where(
        raw_csv_df["HUQ030"] != "",
        np.nan
    ),
    errors="coerce"
)


final_huq041_present = (
    raw_csv_df["HUQ041"]
    .ne("")
    .to_numpy(
        dtype=bool
    )
)


expected_huq041_present = (
    final_huq030
    .isin([1, 3])
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


final_huq041_mismatch = int(
    np.sum(
        expected_huq041_present
        !=
        final_huq041_present
    )
)


final_huq051_numeric = pd.to_numeric(
    raw_csv_df["HUQ051"],
    errors="coerce"
)


expected_huq061_present = (
    final_huq051_numeric
    .eq(0)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


final_huq061_present = (
    raw_csv_df["HUQ061"]
    .ne("")
    .to_numpy(
        dtype=bool
    )
)


final_huq061_mismatch = int(
    np.sum(
        expected_huq061_present
        !=
        final_huq061_present
    )
)


final_huq071 = pd.to_numeric(
    raw_csv_df["HUQ071"],
    errors="coerce"
)


expected_hud080_present = (
    final_huq071
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


final_hud080_present = (
    raw_csv_df["HUD080"]
    .ne("")
    .to_numpy(
        dtype=bool
    )
)


final_hud080_mismatch = int(
    np.sum(
        expected_hud080_present
        !=
        final_hud080_present
    )
)


print(
    "\n--- FINAL ROUTING CHECKS ---"
)

print(
    "HUQ030/HUQ041 mismatches:",
    final_huq041_mismatch
)

print(
    "HUQ051/HUQ061 mismatches:",
    final_huq061_mismatch
)

print(
    "HUQ071/HUD080 mismatches:",
    final_hud080_mismatch
)


if final_huq041_mismatch != 0:

    raise ValueError(
        "Final HUQ030/HUQ041 routing changed."
    )


if final_huq061_mismatch != 0:

    raise ValueError(
        "Final HUQ051/HUQ061 routing changed."
    )


if final_hud080_mismatch != 0:

    raise ValueError(
        "Final HUQ071/HUD080 routing changed."
    )


print(
    "PASS: Final routing relationships preserved."
)


# ============================================================
# 41. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0


for col in EXPECTED_COLUMNS:

    scientific_count += int(
        raw_csv_df[col]
        .str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


print(
    "\n--- SCIENTIFIC NOTATION CHECK ---"
)

print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 42. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 83732:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 93702:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN found in final CSV."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains '.0'."
    )


print(
    "PASS: Final SEQN valid "
    "and contains no '.0'."
)


# ============================================================
# 43. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "HUQ_I CONVERSION COMPLETE"
)

print(
    "=" * 100
)


print(
    "Cycle: 2015-2016"
)


print(
    "Component: Questionnaire - "
    "Hospital Utilization & Access to Care"
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)


print(
    "Original tiny-value artifacts:",
    f"{tiny_total:,}"
)


print(
    "Tiny artifact variable:",
    "HUQ051"
)


print(
    "Validated zero restorations:",
    f"{huq051_zero_count:,}"
)


print(
    "Final HUQ051 zeros:",
    f"{huq051_zero_final:,}"
)


print(
    "Genuine decimal variables:",
    "NONE"
)


print(
    "Whole-number/code variables:",
    len(INTEGER_COLUMNS)
)


print(
    "HUQ010 general health responses:",
    f"{int(df['HUQ010'].notna().sum()):,}"
)


print(
    "HUQ051 healthcare-use responses:",
    f"{int(df['HUQ051'].notna().sum()):,}"
)


print(
    "HUQ061 follow-up responses:",
    f"{int(df['HUQ061'].notna().sum()):,}"
)


print(
    "HUQ071 overnight hospitalization yes:",
    int(
        df["HUQ071"]
        .eq(1)
        .sum()
    )
)


print(
    "HUD080 hospitalization-count responses:",
    f"{int(df['HUD080'].notna().sum()):,}"
)


print(
    "HUQ090 mental-health responses:",
    f"{int(df['HUQ090'].notna().sum()):,}"
)


print(
    "HUQ030/HUQ041 routing mismatches:",
    huq041_routing_mismatch
)


print(
    "HUQ051/HUQ061 routing mismatches:",
    huq061_routing_mismatch
)


print(
    "HUQ071/HUD080 routing mismatches:",
    hud080_routing_mismatch
)


print(
    "XPT -> CSV numeric differences:",
    "0 expected"
)


print(
    "Missing-position differences:",
    "0 expected"
)


print(
    "Tiny values remaining:",
    "0 expected"
)


print(
    "Unnecessary '.0':",
    "0 expected"
)


print(
    "Scientific notation:",
    "0 expected"
)


print(
    "\nFINAL RULE:"
)


print(
    "HUQ_I contains no genuine decimal variables."
)


print(
    "Only the 1,421 validated HUQ051 tiny values "
    "are changed to real numeric zero."
)


print(
    "All 10 variables can then safely be exported "
    "as whole-number IDs/codes."
)


print(
    "Your broader NHANES fidelity rule remains unchanged:"
)


print(
    "Genuine Dietary and other continuous decimal "
    "variables remain exactly AS IS."
)

NHANES 2015-2016 HUQ_I XPT -> CSV
Rows: 9,971
Columns: 10
PASS: Structure = 9,971 rows x 10 columns.

--- FIELD TYPE PLAN ---
Genuine decimal fields: []
String fields: []
Whole-number/code fields: ['SEQN', 'HUQ010', 'HUQ020', 'HUQ030', 'HUQ041', 'HUQ051', 'HUQ061', 'HUQ071', 'HUD080', 'HUQ090']

--- ORIGINAL XPT ZERO CHECK ---
Ordinary numeric zeros before correction: 0
Tiny XPORT artifacts: 1,421
Variables containing tiny artifacts: 1

Tiny values by variable:
HUQ051    1421

PASS: Exact HUQ051 tiny-zero pattern validated.

PASS: 1,421 validated HUQ051 values restored to numeric zero.

Tiny values remaining: 0
PASS: No tiny XPORT artifacts remain.

--- GENUINE DECIMAL DETECTION ---
SEQN: 0 fractional observations
HUQ010: 0 fractional observations
HUQ020: 0 fractional observations
HUQ030: 0 fractional observations
HUQ041: 0 fractional observations
HUQ051: 0 fractional observations
HUQ061: 0 fractional observations
HUQ071: 0 fractional observations
HUD080: 0 fractional observations
HUQ0